# Zero-shot evaluation after debiasing

Evaluates CLIP zero-shot classification on all available CelebA attributes after debiasing a concept.
Compares three debiasing regimes against the raw (undebiased) baseline.

**Regimes:**
- **Raw**: no debiasing — baseline
- **Single**: debiasing hook at one layer L, evaluate final model output
- **Sequential**: all 24 layers debiased sequentially

**Classification** (`zero_shot_classify`): `score = sim(img, pos_text) − sim(img, neg_text)`, predict pos if score > 0.  
Swap `zero_shot_classify` in the dedicated cell to try other approaches.

**Metrics**: balanced accuracy (BA) and AUC-ROC. Main result: ΔBA = BA(debiased) − BA(raw).

Prerequisites: `01_get_text_embeddings.ipynb`, `02_get_activations.ipynb`,
`01_sequential_debiasing.ipynb`, `05_get_single_debiased_activations.ipynb`.

Output: `notebooks/results/zero_shot_eval/{CONCEPT}/`

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT  = 'eyeglasses'   # concept that was debiased
METHODS  = ['diff_means', 'lr', 'pclarc']
SPLIT    = 'test'

# 'all' to evaluate every debias layer, or a list e.g. [0, 6, 12, 18, 23]
SINGLE_DEBIAS_LAYERS = 'all'

NUM_LAYERS = 24

TEXT_EMB_PATH = ROOT / 'data' / 'text_embeddings.parquet'
RAW_DIR       = ROOT / 'data' / 'activations' / 'raw'
DEB_DIR       = ROOT / 'data' / 'activations' / 'debiased'
META_PATH     = ROOT / 'data' / 'metadata.csv'
OUT_DIR       = ROOT / 'notebooks' / 'results' / 'zero_shot_eval' / CONCEPT
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert TEXT_EMB_PATH.exists(), f'Run 01_get_text_embeddings.ipynb first. Missing: {TEXT_EMB_PATH}'
assert RAW_DIR.exists(),       f'Run 02_get_activations.ipynb first. Missing: {RAW_DIR}'
assert META_PATH.exists(),     f'Missing: {META_PATH}'

debias_layers = list(range(NUM_LAYERS)) if SINGLE_DEBIAS_LAYERS == 'all' else SINGLE_DEBIAS_LAYERS

print(f'Concept : {CONCEPT}')
print(f'Methods : {METHODS}')
print(f'Split   : {SPLIT}')
print(f'Output  : {OUT_DIR}')

In [ ]:
df_txt   = pd.read_parquet(TEXT_EMB_PATH)
feat_cols = [str(i) for i in range(768)]

# txt_emb[attribute][label] = (768,) float32 unit vector
txt_emb = {}
for _, row in df_txt.iterrows():
    attr, lbl = row['attribute'], row['label']
    txt_emb.setdefault(attr, {})[lbl] = row[feat_cols].values.astype(np.float32)

# Only keep attributes that have both pos and neg embeddings
attributes = sorted(a for a, d in txt_emb.items() if 'pos' in d and 'neg' in d)
print(f'Loaded text embeddings for {len(attributes)} attributes')

# Load metadata labels (use whatever columns are present in metadata)
df_meta   = pd.read_csv(META_PATH)
df_split  = df_meta[df_meta['split'] == SPLIT].reset_index(drop=True)
meta_cols = [c for c in df_split.columns if c not in ('filename', 'split')]

# Attributes available in both text embeddings and metadata labels
eval_attrs = [a for a in attributes if a in meta_cols]
print(f'Evaluable attributes (text + labels): {len(eval_attrs)}')

In [ ]:
# ── CLASSIFICATION FUNCTION — swap this to try different approaches ───────────
TAU = 100.0  # CLIP-style temperature on cosine similarities


def zero_shot_classify(img_emb_norm, txt_pos, txt_neg):
    """Binary zero-shot classification via softmax over class similarities.

    #10 — instead of `score = sim_pos − sim_neg; pred = score > 0`, build the
    standard CLIP softmax classifier:
        logits = TAU * stack([sim_neg, sim_pos], axis=1)   # column 1 = pos
        probs  = softmax(logits, axis=1)
        pred   = argmax(probs, axis=1)
    The returned `scores` is `probs[:, 1]` (probability of the positive class)
    so AUC / threshold-based metrics can use a calibrated probability.

    Parameters
    ----------
    img_emb_norm : (n, 768) float32  — L2-normalized image embeddings
    txt_pos      : (768,)  float32  — L2-normalized pos-class text embedding
    txt_neg      : (768,)  float32  — L2-normalized neg-class text embedding

    Returns
    -------
    scores : (n,) float32  — P(class = pos)
    preds  : (n,) int      — 0 or 1 (argmax)
    """
    sim_pos = img_emb_norm @ txt_pos
    sim_neg = img_emb_norm @ txt_neg
    logits  = TAU * np.stack([sim_neg, sim_pos], axis=1)
    # numerically-stable softmax
    logits  = logits - logits.max(axis=1, keepdims=True)
    expv    = np.exp(logits)
    probs   = expv / expv.sum(axis=1, keepdims=True)
    preds   = np.argmax(probs, axis=1).astype(int)
    scores  = probs[:, 1].astype(np.float32)
    return scores, preds

In [ ]:
def load_model_output(path):
    """Load model_output.parquet, return (X float32, filenames)."""
    if not path.exists():
        return None, None
    df = pd.read_parquet(path)
    X  = df[feat_cols].values.astype(np.float32)
    return X, df['filename'].tolist()


def l2_normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(norms > 0, norms, 1.0)


def evaluate_condition(X_raw, filenames, label_dict):
    """Run zero_shot_classify for every eval attribute. Returns list of metric dicts."""
    X_norm = l2_normalize(X_raw)
    fn_to_idx = {fn: i for i, fn in enumerate(filenames)}

    results = []
    for attr in eval_attrs:
        if attr not in label_dict:
            continue
        y_true = np.array(label_dict[attr])

        if len(np.unique(y_true)) < 2:
            continue

        scores, preds = zero_shot_classify(X_norm, txt_emb[attr]['pos'], txt_emb[attr]['neg'])

        ba  = balanced_accuracy_score(y_true, preds)
        try:
            auc = roc_auc_score(y_true, scores)
        except Exception:
            auc = float('nan')

        results.append({'target_attribute': attr, 'bal_acc': ba, 'auc': auc})
    return results


def build_label_dict(filenames):
    """Return {attribute: y_array} aligned to filenames, from metadata."""
    df_fn = pd.DataFrame({'filename': filenames}).merge(
        df_split[['filename'] + eval_attrs], on='filename', how='left'
    )
    return {attr: df_fn[attr].values.astype(int) for attr in eval_attrs if attr in df_fn.columns}

In [ ]:
print('=== Raw baseline ===')
X_raw, fn_raw = load_model_output(RAW_DIR / SPLIT / 'model_output.parquet')
assert X_raw is not None, 'Raw model_output.parquet missing — run 02_get_activations.ipynb'

label_dict_raw = build_label_dict(fn_raw)
raw_results    = evaluate_condition(X_raw, fn_raw, label_dict_raw)

df_raw = pd.DataFrame(raw_results)
df_raw['regime'] = 'raw'
df_raw['debiased_concept'] = 'none'
df_raw['method'] = 'none'
df_raw['debias_layer'] = -1

# Build lookup: attribute → raw BA
raw_ba = dict(zip(df_raw['target_attribute'], df_raw['bal_acc']))
print(f'Raw baseline: {len(df_raw)} attributes evaluated')
print(df_raw.sort_values('bal_acc', ascending=False)[['target_attribute', 'bal_acc', 'auc']].head(10))

In [ ]:
records_single = []
print('=== Single debiasing ===')

for method in tqdm(METHODS, desc='Methods'):
    for debias_layer in tqdm(debias_layers, desc=method, leave=False):
        path = (DEB_DIR / 'single' / CONCEPT / method
                / f'from_layer_{debias_layer:02d}' / SPLIT / 'model_output.parquet')
        X, fns = load_model_output(path)
        if X is None:
            continue

        label_dict = build_label_dict(fns)
        for r in evaluate_condition(X, fns, label_dict):
            records_single.append({
                'regime':          'single',
                'debiased_concept': CONCEPT,
                'method':          method,
                'debias_layer':    debias_layer,
                **r,
                'ba_raw':  raw_ba.get(r['target_attribute'], float('nan')),
            })

df_single = pd.DataFrame(records_single)
if not df_single.empty:
    df_single['delta_ba'] = df_single['bal_acc'] - df_single['ba_raw']
print(f'Single: {len(df_single)} rows')

In [ ]:
records_seq = []
print('=== Sequential debiasing ===')

for method in tqdm(METHODS, desc='Methods'):
    path = DEB_DIR / 'sequential' / CONCEPT / method / SPLIT / 'model_output.parquet'
    X, fns = load_model_output(path)
    if X is None:
        continue

    label_dict = build_label_dict(fns)
    for r in evaluate_condition(X, fns, label_dict):
        records_seq.append({
            'regime':           'sequential',
            'debiased_concept': CONCEPT,
            'method':           method,
            'debias_layer':     -1,
            **r,
            'ba_raw': raw_ba.get(r['target_attribute'], float('nan')),
        })

df_seq = pd.DataFrame(records_seq)
if not df_seq.empty:
    df_seq['delta_ba'] = df_seq['bal_acc'] - df_seq['ba_raw']
print(f'Sequential: {len(df_seq)} rows')

In [ ]:
records_iter = []
print('=== Iterative single-layer debiasing ===')

iterative_base = DEB_DIR / 'iterative' / CONCEPT
for method in tqdm(METHODS, desc='Methods'):
    method_dir = iterative_base / method
    if not method_dir.exists():
        continue
    for layer_dir in sorted(method_dir.iterdir()):
        if not layer_dir.name.startswith('layer_'):
            continue
        debias_layer = int(layer_dir.name.split('_')[1])
        path = layer_dir / SPLIT / 'model_output.parquet'
        X, fns = load_model_output(path)
        if X is None:
            continue

        n_iter = None
        info_path = layer_dir / 'info.json'
        if info_path.exists():
            with open(info_path) as f:
                n_iter = json.load(f).get('n_iterations')

        label_dict = build_label_dict(fns)
        for r in evaluate_condition(X, fns, label_dict):
            records_iter.append({
                'regime':           'iterative',
                'debiased_concept': CONCEPT,
                'method':           method,
                'debias_layer':     debias_layer,
                'n_iterations':     n_iter,
                **r,
                'ba_raw': raw_ba.get(r['target_attribute'], float('nan')),
            })

df_iter = pd.DataFrame(records_iter)
if not df_iter.empty:
    df_iter['delta_ba'] = df_iter['bal_acc'] - df_iter['ba_raw']
print(f'Iterative: {len(df_iter)} rows')

In [ ]:
records_fixed = []
print('=== Fixed-direction multi-layer debiasing ===')

fixed_base = DEB_DIR / 'fixed_multi' / CONCEPT
for method in tqdm(METHODS, desc='Methods'):
    method_dir = fixed_base / method
    if not method_dir.exists():
        continue
    for run_dir in sorted(method_dir.iterdir()):
        path = run_dir / SPLIT / 'model_output.parquet'
        X, fns = load_model_output(path)
        if X is None:
            continue

        start_layer = n_layers = None
        info_path = run_dir / 'info.json'
        if info_path.exists():
            with open(info_path) as f:
                info = json.load(f)
                start_layer = info.get('start_layer')
                n_layers    = info.get('n_layers')

        label_dict = build_label_dict(fns)
        for r in evaluate_condition(X, fns, label_dict):
            records_fixed.append({
                'regime':           'fixed_multi',
                'debiased_concept': CONCEPT,
                'method':           method,
                'debias_layer':     start_layer,
                'n_layers':         n_layers,
                **r,
                'ba_raw': raw_ba.get(r['target_attribute'], float('nan')),
            })

df_fixed = pd.DataFrame(records_fixed)
if not df_fixed.empty:
    df_fixed['delta_ba'] = df_fixed['bal_acc'] - df_fixed['ba_raw']
print(f'Fixed multi: {len(df_fixed)} rows')

In [ ]:
df_all = pd.concat([df_raw, df_single, df_seq, df_iter, df_fixed], ignore_index=True)
df_all.to_csv(OUT_DIR / 'results.csv', index=False)
print(f'Saved {len(df_all)} rows to results.csv')
print(df_all['regime'].value_counts().to_string())

In [ ]:
# ## Visualization

In [ ]:
# ΔBA on the debiased concept itself — does debiasing actually reduce detectability?

fig, ax = plt.subplots(figsize=(10, 4))

if not df_single.empty:
    sub = df_single[df_single['target_attribute'] == CONCEPT]
    for method in METHODS:
        s = sub[sub['method'] == method].sort_values('debias_layer')
        if not s.empty:
            ax.plot(s['debias_layer'], s['delta_ba'], marker='o', ms=4, label=f'single/{method}')

if not df_seq.empty:
    sub = df_seq[df_seq['target_attribute'] == CONCEPT]
    for method in METHODS:
        s = sub[sub['method'] == method]
        if not s.empty:
            ax.axhline(s['delta_ba'].values[0], ls='--', label=f'seq/{method}')

ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_xlabel('Debias layer')
ax.set_ylabel('ΔBA (debiased − raw)')
ax.set_title(f'Effect on debiased concept: {CONCEPT}')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(OUT_DIR / f'delta_ba_debiased_concept.png', dpi=150)
plt.show()

In [ ]:
# ΔBA on all attributes for sequential debiasing (collateral damage overview)

if not df_seq.empty:
    fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 6), sharey=True)
    if len(METHODS) == 1:
        axes = [axes]

    for ax, method in zip(axes, METHODS):
        sub = df_seq[df_seq['method'] == method].sort_values('delta_ba')
        if sub.empty:
            continue
        colors = ['tomato' if d < 0 else 'steelblue' for d in sub['delta_ba']]
        ax.barh(sub['target_attribute'], sub['delta_ba'], color=colors)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'Sequential — {method}', fontsize=10)
        ax.set_xlabel('ΔBA')

    fig.suptitle(f'ΔBA across all attributes — debiased concept: {CONCEPT}', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'delta_ba_all_attrs_sequential.png', dpi=150)
    plt.show()

In [ ]:
# Heatmap: ΔBA per (target_attribute × debias_layer) for single debiasing, one plot per method

if not df_single.empty:
    for method in METHODS:
        sub = df_single[df_single['method'] == method]
        if sub.empty:
            continue
        pivot = sub.pivot_table(
            index='target_attribute', columns='debias_layer', values='delta_ba', aggfunc='mean'
        )
        vmax = max(abs(pivot.values[np.isfinite(pivot.values)]).max(), 0.01)

        fig, ax = plt.subplots(figsize=(14, max(4, len(pivot) * 0.35)))
        sns.heatmap(
            pivot, ax=ax, cmap='RdBu_r', center=0,
            vmin=-vmax, vmax=vmax, annot=False, linewidths=0.2,
        )
        ax.set_title(f'ΔBA (debiased−raw) — single, method={method}, concept={CONCEPT}', fontsize=10)
        ax.set_xlabel('Debias layer')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'heatmap_delta_ba_single_{method}.png', dpi=150)
        plt.show()

In [ ]:
# Raw BA per attribute — how well does undebiased CLIP classify each attribute?

df_raw_sorted = df_raw.sort_values('bal_acc', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(4, len(df_raw_sorted) * 0.3)))
colors = ['gold' if a == CONCEPT else 'steelblue' for a in df_raw_sorted['target_attribute']]
ax.barh(df_raw_sorted['target_attribute'], df_raw_sorted['bal_acc'], color=colors)
ax.axvline(0.5, color='red', lw=0.8, ls='--', label='random (BA=0.5)')
ax.set_xlabel('Balanced accuracy')
ax.set_title('Raw CLIP zero-shot BA per attribute (gold = debiased concept)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'raw_ba_per_attribute.png', dpi=150)
plt.show()